In [0]:


# Read CSV files with schema evolution handling
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mergeSchema", "true")
    .csv("/path/to/csv/files/")
)

display(df)

# Data Pipeline![image_1782823871688.png](./image_1782823871688.png "image_1782823871688.png")![image_1782823890078.png](./image_1782823890078.png "image_1782823890078.png")



In [0]:

# Example: Schema evolution in CSV files

# Assume /path/to/csv/files/ contains:
# file1.csv: id,name
# file2.csv: id,name,age

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mergeSchema", "true")
    .csv("/path/to/csv/files/")
)

display(df)

In [0]:
This code demonstrates **schema evolution** when reading multiple CSV files with potentially different schemas using Spark.

## What It Does

The code reads CSV files from a directory where files may have different columns:
* **file1.csv** has columns: `id`, `name`
* **file2.csv** has columns: `id`, `name`, `age`

## Key Options Explained

* **`option("header", "true")`** – Treats the first row of each CSV file as column names

* **`option("inferSchema", "true")`** – Automatically detects data types (e.g., integers, strings) instead of reading everything as strings

* **`option("mergeSchema", "true")`** – This is the critical option for schema evolution. It:
  - Reads all CSV files in the directory
  - Combines different column sets from all files
  - Creates a unified schema with **all columns** found across files
  - Fills missing columns with `null` values where they don't exist

## Result

The final DataFrame `df` will have **all three columns** (`id`, `name`, `age`):
* Rows from file1.csv will have `null` values in the `age` column
* Rows from file2.csv will have values for all three columns

## Use Case

This is useful when:
* Files are added incrementally with new fields over time
* You want to read all data without failing on schema mismatches
* You need backward compatibility as your data structure evolves

Without `mergeSchema`, Spark would infer the schema from one file and fail or drop columns when encountering files with different structures.

![image_1782824066060.png](./image_1782824066060.png "image_1782824066060.png")



![image_1782824177618.png](./image_1782824177618.png "image_1782824177618.png")

| Feature             | `mergeSchema`                      | `overwriteSchema`                            |
| ------------------- | ---------------------------------- | -------------------------------------------- |
| Adds new columns    | ✅ Yes                              | ✅ Yes                                        |
| Removes columns     | ❌ No                               | ✅ Yes                                        |
| Renames columns     | ❌ No                               | ✅ Yes (by replacing the schema)              |
| Changes data types  | ❌ Usually not                      | ✅ Yes (when overwriting)                     |
| Keeps existing data | ✅ Yes (append/evolution scenarios) | ❌ No (overwrite replaces the table contents) |
| Typical mode        | Append                             | Overwrite                                    |




# Understanding `overwriteSchema` in Spark

`overwriteSchema` is used when you want to **replace the existing table schema** with the schema of the DataFrame you are writing.

Unlike `mergeSchema`, which **adds new columns**, `overwriteSchema` **completely replaces the schema**.

> **Note:** `overwriteSchema` is typically used with **`mode("overwrite")`**.

---

# Syntax

```python
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/delta/employees")
```

Or when writing to a managed table:

```python
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("employees")
```

---

# Example

## Existing Delta Table

| id | name | salary |
|----|------|---------|
| 1 | Alice | 50000 |

### Existing Schema

```text
id INT
name STRING
salary DOUBLE
```

---

## New DataFrame

Suppose the business redesigns the employee table.

```text
id INT
full_name STRING
department STRING
```

Notice the changes:

- `name` → `full_name`
- `salary` removed
- `department` added

---

# Without `overwriteSchema`

```python
df.write \
    .mode("overwrite") \
    .saveAsTable("employees")
```

### Result

```
AnalysisException:
Schema mismatch detected.
```

Spark prevents the overwrite because the new schema is different.

---

# With `overwriteSchema`

```python
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("employees")
```

### New Table

| id | full_name | department |
|----|------------|------------|
| 1 | Alice | HR |

The old schema is completely replaced.

---

# When to Use `overwriteSchema`

Use `overwriteSchema` when you intentionally want to replace the table structure.

### Common Scenarios

- Complete schema redesign
- Column renaming
- Column removal
- Data type changes
- Rebuilding reporting tables
- Development and testing

---

# Real-Time Examples

## Example 1: Source System Redesign

### Old Source

| customer_id | name | age |
|--------------|------|-----|
| 101 | Alice | 30 |

### New Source

| customer_id | first_name | last_name | email |
|--------------|------------|-----------|-------|
| 101 | Alice | Smith | alice@email.com |

The source application has changed completely.

Instead of altering the existing table, overwrite it with the new schema.

```python
df.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .save("/delta/customer")
```

---

## Example 2: Reporting Table Redesign

### Old Table

```text
order_id
amount
```

### New Table

```text
order_id
net_amount
tax
discount
```

Business requirements changed.

Instead of adding and dropping columns manually, overwrite the table.

---

## Example 3: Data Type Change

### Existing Schema

```text
salary DOUBLE
```

### Required Schema

```text
salary DECIMAL(18,2)
```

Since the column type changes, `mergeSchema` cannot help.

Use:

```python
df.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .save("/delta/employees")
```

---

## Example 4: Development Environment

During development, schemas change frequently.

Instead of dropping tables manually every time:

```python
df.write \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("employee_dev")
```

This recreates the table with the latest schema.

---

# `mergeSchema` vs `overwriteSchema`

| Feature | `mergeSchema` | `overwriteSchema` |
|----------|---------------|-------------------|
| Adds new columns | ✅ Yes | ✅ Yes |
| Removes columns | ❌ No | ✅ Yes |
| Renames columns | ❌ No | ✅ Yes |
| Changes data types | ❌ No | ✅ Yes |
| Keeps existing schema | ✅ Yes | ❌ No |
| Requires overwrite mode | ❌ No | ✅ Yes |
| Typical use | Schema evolution | Schema replacement |

---

# Interview Scenario

### Existing Table

```text
id
name
salary
```

### New DataFrame

```text
id
full_name
department
```

### Which option should you use?

❌ `mergeSchema`

Reason:
- It only adds new columns.
- It does not remove or rename existing columns.

✅ `overwriteSchema`

Reason:
- It replaces the existing schema with the new one.
- Supports renamed columns, removed columns, and structural changes.

---

# `mergeSchema` vs `overwriteSchema` Decision Guide

| Situation | Recommended Option |
|------------|--------------------|
| Add a new column | ✅ `mergeSchema` |
| Append new data with extra columns | ✅ `mergeSchema` |
| Read files with different schemas | ✅ `mergeSchema` |
| Rename a column | ✅ `overwriteSchema` |
| Remove a column | ✅ `overwriteSchema` |
| Change a data type | ✅ `overwriteSchema` |
| Completely redesign a table | ✅ `overwriteSchema` |

---

# Important Notes

> `overwriteSchema` replaces the table schema **only when using** `mode("overwrite")`.

> It is intended for **planned schema changes**, not for routine data appends.

> If you overwrite a table, the existing data is replaced by the new DataFrame unless you use advanced techniques such as partition overwrite.

---

# Key Takeaways

- ✅ Use **`mergeSchema`** for schema evolution (adding new columns).
- ✅ Use **`overwriteSchema`** for schema replacement (renaming, removing columns, changing data types).
- ✅ `overwriteSchema` is almost always used with **`mode("overwrite")`**.
- ❌ Do not use `overwriteSchema` when you simply want to append new columns.
- 🚀 Think of `overwriteSchema` as recreating the table with a new structure.


# Understanding `mergeSchema` in Spark

`mergeSchema` is used when **schemas differ** between your DataFrame and the existing data/table. It can be used **while reading** or **while writing**, but the purpose is different in each case.

---

# 1. Using `mergeSchema` While Reading

Use `mergeSchema` when **existing files already have different schemas**, and you want Spark to combine them into a single schema.

## Example

Suppose your Parquet folder contains two files.

### File 1

| id | name |
|----|------|
| 1 | Alice |

### File 2

| id | name | age |
|----|------|-----|
| 2 | Bob | 30 |

### Without `mergeSchema`

```python
df = spark.read.parquet("/data/employees")
```

Spark may infer the schema from only one set of files, so the `age` column might not appear.

### With `mergeSchema`

```python
df = spark.read \
    .option("mergeSchema", "true") \
    .parquet("/data/employees")
```

### Output

| id | name | age |
|----|------|-----|
| 1 | Alice | NULL |
| 2 | Bob | 30 |

### When to use

- Existing Parquet/Delta files have different schemas.
- Historical data was written with older schemas.
- You want Spark to automatically combine all columns while reading.

> **Note:** Reading with `mergeSchema` **does not change the table schema**. It only affects how Spark reads the data.

---

# 2. Using `mergeSchema` While Writing

Use `mergeSchema` when **appending data that contains new columns** to an existing Delta table.

## Existing Table

| id | name |
|----|------|
| 1 | Alice |

## New DataFrame

| id | name | department |
|----|------|------------|
| 2 | Bob | IT |

### Without `mergeSchema`

```python
df.write \
  .mode("append") \
  .format("delta") \
  .save("/delta/employees")
```

**Result**

```
AnalysisException:
Schema mismatch detected.
```

### With `mergeSchema`

```python
df.write \
  .option("mergeSchema", "true") \
  .mode("append") \
  .format("delta") \
  .save("/delta/employees")
```

### New Table Schema

| id | name | department |
|----|------|------------|
| 1 | Alice | NULL |
| 2 | Bob | IT |

### When to use

- Appending new columns to an existing Delta table.
- Allowing schema evolution.
- Avoiding schema mismatch errors during append operations.

> **Note:** Writing with `mergeSchema` updates the **table metadata** by adding new columns.

---

# Reading vs Writing

| Scenario | Use `mergeSchema`? |
|-----------|--------------------|
| Existing files have different schemas | ✅ While Reading |
| Appending new columns to a Delta table | ✅ While Writing |
| All files have the same schema | ❌ Not Required |
| Changing data types | ❌ Use `overwriteSchema` instead |
| Renaming or removing columns | ❌ Use `overwriteSchema` instead |

---

# Real-Time Example

## Scenario

Every month, the HR system sends employee data.

### January

| id | name |
|----|------|
| 1 | Alice |

### February

The HR team adds a new column.

| id | name | location |
|----|------|----------|
| 2 | Bob | Pune |

### Solution

Use `mergeSchema` while writing.

```python
new_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/delta/employees")
```

The table becomes:

| id | name | location |
|----|------|----------|
| 1 | Alice | NULL |
| 2 | Bob | Pune |

Older records automatically get `NULL` for the new column.

---

# Interview Question

### Q: Why use `mergeSchema` while writing if we can use it while reading?

### Answer

**Reading `mergeSchema`:**
- Combines schemas from existing files.
- Does **not** update the table schema.
- Only affects the current read operation.

**Writing `mergeSchema`:**
- Updates the table schema.
- Adds new columns permanently.
- Supports schema evolution for future reads and writes.

---

# Summary

| Feature | Read `mergeSchema` | Write `mergeSchema` |
|----------|--------------------|---------------------|
| Combines schemas | ✅ Yes | ❌ No |
| Updates table schema | ❌ No | ✅ Yes |
| Adds new columns | Reads them only | Adds them permanently |
| Common use case | Reading historical files | Appending new columns |
| Changes existing data | ❌ No | ❌ No |

---

# Key Takeaways

- ✅ Use **`mergeSchema` while reading** when existing files have different schemas.
- ✅ Use **`mergeSchema` while writing** when appending data with new columns to a Delta table.
- ❌ Do **not** use `mergeSchema` for renaming columns, removing columns, or changing data types.
- ✅ For schema replacement, use **`overwriteSchema`** with `mode("overwrite")`.